# Lab 07: Evaluations

## Overview

In this final notebook, we run comprehensive evaluations across all deployed agent versions to validate that optimizations improved cost and latency **without degrading quality**.

**What you'll learn:**
- How to run systematic evaluations across agent versions
- How to compare metrics side-by-side
- How to validate quality hasn't degraded
- How to generate a final optimization report

## Prerequisites

- Completed Labs 01-06 (all agent versions deployed)

## Workshop Journey

```
01 Baseline → 02 Quick Wins → 03 Caching → 04 Routing → 05 Guardrails → 06 Gateway → [07 Evaluations]
                                                                                          ↑
                                                                                     You are here
```

## Agent Evaluation — The Bigger Picture

Evaluating an AI agent goes far beyond measuring cost and speed. In practice, agent quality evaluation spans multiple dimensions — and you have plenty of options for how to measure them.

**Response quality** asks whether the agent's answers are actually helpful, correct, and relevant. Did it faithfully use the retrieved information, or did it hallucinate? Was the response concise, or did it ramble? Did it follow the system prompt instructions?

**Tool use correctness** evaluates whether the agent selected the right tool for the job and passed the correct parameters. For our customer support agent: did it call `get_return_policy` when asked about returns (not `get_product_info`), and did it extract the right product category?

**Safety and compliance** checks whether the agent refused inappropriate requests, avoided harmful content, and didn't exhibit bias. Red-teaming and adversarial evaluation systematically probe for failure modes before real users find them.

**End-to-end goal completion** looks at the full conversation: did the agent actually solve the user's problem?

There is a rich and growing ecosystem of open-source frameworks and managed services you can choose from to evaluate these dimensions — [RAGAS](https://docs.ragas.io/) for RAG-specific metrics like faithfulness and context relevance, [DeepEval](https://docs.confident-ai.com/) for full-spectrum evaluation including tool correctness and safety red-teaming, [Evidently AI](https://www.evidentlyai.com/) for LLM-as-a-judge and production monitoring, [Giskard](https://docs.giskard.ai/) for vulnerability scanning, and the [Langfuse](https://langfuse.com/docs) platform you've been using throughout this workshop which also supports evaluation scores. On AWS, [Amazon Bedrock AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html) provides 14 built-in evaluators — from helpfulness and correctness at the trace level, to tool selection and parameter accuracy at the tool level, to goal success rate at the session level. All of these are available to you, and you're free to choose whichever combination fits your use case.

**In this notebook**, we start with **operational metrics**: cost, latency, and token usage across all agent versions, validating that our optimizations reduced cost and latency without breaking things. Then, in **Step 8**, we add a focused **quality evaluation** using three [RAGAS](https://docs.ragas.io/) metrics — **Context Precision**, **Answer Correctness**, and **Tool Call Accuracy** — to confirm that optimization didn't degrade the agent's actual behavior. The quality scores are pushed back to Langfuse so they sit alongside the operational metrics on each trace.

## Step 1: Setup

In [ ]:
from __future__ import annotations

import json
import os
import time
import uuid

from dotenv import load_dotenv

load_dotenv(override=True)

import boto3
import pandas as pd

region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
control_client = boto3.client("bedrock-agentcore-control", region_name=region)
data_client = boto3.client("bedrock-agentcore", region_name=region)

print(f"Region: {region}")
print(f"Langfuse Host: {os.environ.get('LANGFUSE_BASE_URL', 'https://cloud.langfuse.com')}")

## Step 2: Find All Deployed Agent Versions

In [ ]:
def find_agent_by_name(name_pattern):
    """Find agent ARN by name pattern (handles both hyphen and underscore naming)."""
    response = control_client.list_agent_runtimes()
    agents = response.get("agentRuntimes", [])
    for agent in agents:
        agent_name = agent["agentRuntimeName"]
        # Handle both hyphen and underscore naming conventions
        if name_pattern.replace("-", "_") in agent_name or name_pattern in agent_name:
            return agent["agentRuntimeArn"]
    return None


# Find all agent versions
versions = {
    "v1-baseline": find_agent_by_name("v1-baseline"),
    "v2-quick-wins": find_agent_by_name("v2-quick-wins"),
    "v3-caching": find_agent_by_name("v3-caching"),
    "v4-routing": find_agent_by_name("v4-routing"),
    "v5-guardrails": find_agent_by_name("v5-guardrails"),
    "v6-gateway": find_agent_by_name("v6-gateway"),
}

print("Found agent versions:")
for name, arn in versions.items():
    status = "Found" if arn else "Not found"
    print(f"  {name}: {status}")

## Step 3: Load Test Scenarios

In [ ]:
# Standard test prompts - each demonstrates a specific tool usage pattern
TEST_PROMPTS = [
    # Single tool: get_return_policy
    {"id": "return-policy", "query": "What is your return policy for laptops?"},
    # Single tool: get_product_info
    {"id": "product-info", "query": "Tell me about your smartphone options"},
    # Single tool: get_technical_support (Bedrock KB)
    {"id": "technical-support", "query": "My laptop won't turn on, can you help me troubleshoot?"},
    # Multi-tool: get_product_info + get_return_policy
    {"id": "multi-part", "query": "I want to buy a laptop. What are the specs and what's the return policy?"},
    # No tool: General greeting
    {"id": "general", "query": "Hello! What can you help me with today?"},
]

test_scenarios = TEST_PROMPTS

print(f"Loaded {len(test_scenarios)} test scenarios:")
for scenario in test_scenarios:
    print(f"  - {scenario['id']}: {scenario['query'][:50]}...")

In [ ]:
# Import Langfuse metrics helper
from utils.langfuse_metrics import clear_metrics, get_latest_trace_metrics

print("Langfuse metrics helper imported")

## Step 4: Run Evaluations

In [ ]:
# Map version name to agent trace name (uses hyphens)
TRACE_NAME_MAP = {
    "v1-baseline": "customer-support-v1-baseline",
    "v2-quick-wins": "customer-support-v2-quick-wins",
    "v3-caching": "customer-support-v3-caching",
    "v4-routing": "customer-support-v4-routing",
    "v5-guardrails": "customer-support-v5-guardrails",
    "v6-gateway": "customer-support-v6-gateway",
}


def invoke_agent(agent_arn, prompt):
    """Invoke agent and measure latency."""
    start_time = time.time()
    response = data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    latency_ms = (time.time() - start_time) * 1000
    result = json.loads(response["response"].read().decode("utf-8"))
    return result, latency_ms


def run_evaluation(version_name, agent_arn, scenarios):
    """Run all scenarios against an agent version and collect Langfuse metrics."""
    results = []
    total_latency = 0
    successful = 0
    langfuse_metrics_list = []

    agent_trace_name = TRACE_NAME_MAP.get(version_name, version_name)

    print(f"\nEvaluating {version_name}...")
    clear_metrics()  # Clear for this version

    for scenario in scenarios:
        try:
            _result, latency = invoke_agent(agent_arn, scenario["query"])

            # Fetch Langfuse metrics for this trace
            metrics = get_latest_trace_metrics(
                agent_name=agent_trace_name,
                wait_seconds=3,
                max_retries=3,
                timeout_seconds=60,
            )
            langfuse_metrics_list.append(metrics)

            results.append(
                {
                    "scenario_id": scenario["id"],
                    "success": True,
                    "latency_ms": latency,
                }
            )
            total_latency += latency
            successful += 1
            print(f"  [{scenario['id']}] {latency:.0f}ms")
        except Exception as e:
            results.append(
                {
                    "scenario_id": scenario["id"],
                    "success": False,
                    "error": str(e),
                }
            )
            langfuse_metrics_list.append({"error": str(e)})
            print(f"  [{scenario['id']}] FAILED: {e}")

    return {
        "version": version_name,
        "results": results,
        "total_scenarios": len(scenarios),
        "successful": successful,
        "avg_latency_ms": total_latency / successful if successful > 0 else 0,
        "langfuse_metrics": langfuse_metrics_list,
    }

In [ ]:
# Run evaluations for all available versions
all_results = {}

for version_name, agent_arn in versions.items():
    if agent_arn:
        all_results[version_name] = run_evaluation(version_name, agent_arn, test_scenarios)
    else:
        print(f"\nSkipping {version_name} (not deployed)")

## Step 5: Generate Comparison Report

**A note on sample size:** This evaluation runs 5 test queries per version. With a small sample, individual runs may show variance — for example, one version might appear slightly slower or costlier than expected due to network jitter, cold starts, or cache timing. Don't over-index on small differences between adjacent versions (e.g., v4 vs v5).

What to look for:
- **Overall trend** from v1 → v6: cost and latency should generally decrease
- **Large, consistent gains** (e.g., caching in v3, routing in v4) will be visible even at small scale
- **Small or marginal differences** between versions may not be statistically significant

In production, you would run a larger evaluation set (50-100+ queries) to get stable, reliable comparisons with tighter confidence intervals.

In [ ]:
# Create comparison DataFrame
rows = []
for version_name, eval_result in all_results.items():
    success_rate = (eval_result["successful"] / eval_result["total_scenarios"]) * 100

    # Use Langfuse latency (server-side) instead of client-side E2E latency
    lf_metrics = eval_result.get("langfuse_metrics", [])
    valid = [m for m in lf_metrics if "error" not in m]
    avg_latency_s = sum(m.get("latency_seconds", 0) or 0 for m in valid) / len(valid) if valid else 0

    rows.append(
        {
            "Version": version_name,
            "Success Rate": f"{success_rate:.0f}%",
            "Avg Latency (s)": f"{avg_latency_s:.2f}",
            "Successful": eval_result["successful"],
            "Failed": eval_result["total_scenarios"] - eval_result["successful"],
        }
    )

df = pd.DataFrame(rows)
print("\n" + "=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)
print(df.to_string(index=False))

In [ ]:
# Show Langfuse metrics for each version
print("\n" + "=" * 100)
print("LANGFUSE METRICS BY VERSION")
print("=" * 100)

version_summaries = []
for version_name, eval_result in all_results.items():
    lf_metrics = eval_result.get("langfuse_metrics", [])
    valid = [m for m in lf_metrics if "error" not in m]

    if valid:
        total_input = sum(m.get("input_tokens", 0) for m in valid)
        total_output = sum(m.get("output_tokens", 0) for m in valid)
        total_cache_read = sum(m.get("cache_read_tokens", 0) for m in valid)
        total_cache_write = sum(m.get("cache_write_tokens", 0) for m in valid)
        total_cost = sum(m.get("cost_usd", 0) for m in valid)
        avg_latency = sum(m.get("latency_seconds", 0) or 0 for m in valid) / len(valid)
        avg_input = total_input / len(valid)

        version_summaries.append(
            {
                "Version": version_name,
                "Avg Input Tokens": f"{avg_input:,.0f}",
                "Avg Latency (s)": f"{avg_latency:.2f}",
                "Total Cost": f"${total_cost:.4f}",
                "Cache Read": f"{total_cache_read:,}",
                "Cache Write": f"{total_cache_write:,}",
            }
        )

if version_summaries:
    df_langfuse = pd.DataFrame(version_summaries)
    print(df_langfuse.to_string(index=False))

In [ ]:
# Calculate comprehensive improvement from baseline (including token metrics)
if "v1-baseline" in all_results and len(all_results) > 1:
    baseline = all_results["v1-baseline"]

    # Get baseline Langfuse metrics
    baseline_lf = baseline.get("langfuse_metrics", [])
    baseline_valid = [m for m in baseline_lf if "error" not in m]
    baseline_input = (
        sum(m.get("input_tokens", 0) for m in baseline_valid) / len(baseline_valid) if baseline_valid else 0
    )
    baseline_cost = sum(m.get("cost_usd", 0) for m in baseline_valid)
    baseline_latency = (
        sum(m.get("latency_seconds", 0) or 0 for m in baseline_valid) / len(baseline_valid)
        if baseline_valid
        else 0
    )

    print("\n" + "=" * 80)
    print("IMPROVEMENT VS BASELINE (v1)")
    print("=" * 80)
    print(f"{'Version':<20} {'Input Tokens':<18} {'Latency':<15} {'Cost':<15}")
    print("-" * 80)
    print(f"{'v1-baseline':<20} {'(baseline)':<18} {'(baseline)':<15} {'(baseline)':<15}")

    for version_name, eval_result in all_results.items():
        if version_name == "v1-baseline":
            continue

        lf_metrics = eval_result.get("langfuse_metrics", [])
        valid = [m for m in lf_metrics if "error" not in m]

        if valid and baseline_input > 0:
            avg_input = sum(m.get("input_tokens", 0) for m in valid) / len(valid)
            total_cost = sum(m.get("cost_usd", 0) for m in valid)
            avg_latency = sum(m.get("latency_seconds", 0) or 0 for m in valid) / len(valid)

            token_change = ((baseline_input - avg_input) / baseline_input) * 100
            latency_change = (
                ((baseline_latency - avg_latency) / baseline_latency) * 100 if baseline_latency > 0 else 0
            )
            cost_change = ((baseline_cost - total_cost) / baseline_cost) * 100 if baseline_cost > 0 else 0

            print(
                f"{version_name:<20} {token_change:+.1f}%{'':<12} {latency_change:+.1f}%{'':<10} {cost_change:+.1f}%"
            )
        else:
            print(f"{version_name:<20} {'N/A':<18} {'N/A':<15} {'N/A':<15}")

    print("=" * 80)
    print("\nPositive % = improvement (reduction), Negative % = regression (increase)")

## Step 6: View Langfuse Dashboard

In [ ]:
langfuse_base_url = os.environ.get("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")
print(f"View comprehensive metrics at: {langfuse_base_url}")
print("\nFor detailed comparison:")
print("1. Filter by version tags (v1-baseline, v2-quick-wins, etc.)")
print("2. Compare token usage across versions")
print("3. Check cache hit rates (v3+)")
print("4. Verify model routing (v4+)")
print("5. Review guardrail interventions (v5+)")

## Step 7: Final Summary

In [ ]:
print("\n" + "=" * 70)
print("WORKSHOP SUMMARY")
print("=" * 70)
print("""
Optimizations Applied:

1. Quick Wins (v2)
   - Concise system prompt: ~60% token reduction
   - max_tokens limit: Bounded output
   - stop_sequences: Early termination

2. Prompt Caching (v3)
   - System prompt caching: 90% discount on repeated requests
   - Tool definition caching: Additional savings

3. Model Routing (v4)
   - Haiku for simple queries: 12x cheaper input tokens
   - Sonnet for complex queries: Maintained quality

4. Guardrails (v5)
   - Topic filtering: Block off-topic queries
   - Content filtering: Improve safety
   - Zero LLM tokens for blocked queries

5. Gateway (v6)
   - Semantic tool search: Load only relevant tools
   - Reduced context size: Up to 75% fewer tool tokens

Success Criteria:
- Cost should decrease from v1 to v6
- Latency should improve from v1 to v6
- Quality (success rate) should NOT degrade
""")

## Step 8: Quality Evaluation with RAGAS

The operational metrics above show optimization made the agent cheaper and faster. But the workshop's success criterion is "cost and latency down **without degrading quality**." This section measures quality directly with three [RAGAS](https://docs.ragas.io/) metrics:

| Metric | Question it answers | Applies to |
|---|---|---|
| **Tool Call Accuracy** | Did the agent select the right tool(s)? | All scenarios with an expected tool (name-only match) |
| **Answer Correctness** | Is the answer factually right vs. a known-good reference? | Scenarios with an authored `reference` |
| **Context Precision** | Are the *retrieved* KB chunks relevant? | RAG scenarios only (`get_technical_support`) |

**Why these three apply to different scenarios:** Context Precision is a *retrieval* metric — it only makes sense where the agent actually retrieves (the `get_technical_support` tool over a Bedrock Knowledge Base). The other tools return data from fixed lookups, not retrieval. Knowing *when* a metric applies is itself an evaluation skill.

**How we run it.** The deployed agents return only their final text — the intermediate tool calls and retrieved chunks aren't in the response payload, and aren't reliably in the traces either. So we run the agents **locally in-notebook**, capturing tool calls and retrieved contexts directly from the agent's message history. We compare **v1 (baseline)** against **v4 (optimized)** — v4 carries the optimized prompt and all four tools, which is the answer-quality configuration the later versions build on. (v6's gateway and v5's guardrails are runtime concerns that don't change answer correctness, and v6 can't run locally without a live gateway.)

**Judge model.** RAGAS uses an LLM-as-judge (Context Precision, Answer Correctness) plus embeddings (Answer Correctness similarity). Both run on **Amazon Bedrock** — Claude Sonnet 4.5 as judge, Titan Text v2 for embeddings — so no extra API keys are needed.

Finally, every score is pushed back to **Langfuse** as a numeric score on the agent's trace, so quality and operational metrics live together.

In [ ]:
# Install RAGAS eval dependencies (pinned -- see requirements-eval.txt for why).
# Quality eval runs the agents locally, so these are notebook-only deps.
%pip install -q -r requirements-eval.txt
print("RAGAS eval dependencies installed. Restart the kernel if imports fail.")

In [ ]:
# Build the two local agents and the Bedrock RAGAS judge.
import json

from strands import Agent
from strands.models import BedrockModel
from strands.types.content import SystemContentBlock

from agents.v1_baseline import SYSTEM_PROMPT as V1_PROMPT  # verbose, unoptimized baseline prompt
from utils import ragas_eval
from utils.agent_config import MODEL_SONNET, SYSTEM_PROMPT_TEXT
from utils.tools import get_product_info, get_return_policy, get_technical_support, web_search

TOOLS = [get_return_policy, get_product_info, web_search, get_technical_support]


def build_v1_agent():
    """Baseline: verbose prompt, Sonnet, no optimizations."""
    return Agent(
        model=BedrockModel(model_id=MODEL_SONNET, temperature=0.3, region_name=region),
        tools=TOOLS,
        system_prompt=V1_PROMPT,
        name="customer-support-v1-baseline",
        trace_attributes={"version": "v1-baseline", "langfuse.tags": ["ragas-eval", "v1-baseline"]},
    )


def build_v4_agent():
    """Optimized proxy: concise prompt + caching + bounded output, all 4 tools loaded directly.

    This is v4/v6's answer-quality configuration without the gateway plumbing.
    Uses Sonnet for both v1 and v4 so the comparison isolates prompt quality, not model tier.
    """
    return Agent(
        model=BedrockModel(
            model_id=MODEL_SONNET,
            temperature=0.1,
            max_tokens=1024,
            stop_sequences=["###", "END_RESPONSE"],
            cache_tools="default",
            region_name=region,
        ),
        tools=TOOLS,
        system_prompt=[
            SystemContentBlock(text=SYSTEM_PROMPT_TEXT),
            SystemContentBlock(cachePoint={"type": "default"}),
        ],
        name="customer-support-v4-optimized",
        trace_attributes={"version": "v4-optimized", "langfuse.tags": ["ragas-eval", "v4-optimized"]},
    )


# Bedrock judge + embeddings for RAGAS (Sonnet judge, Titan v2 embeddings).
judge, embeddings = ragas_eval.build_bedrock_evaluator(region=region)
print("Built v1 + v4 agents and Bedrock RAGAS judge.")

In [ ]:
# Load the labeled scenarios (only those carrying a ground-truth `reference`).
with open("data/test_scenarios.json", encoding="utf-8") as f:
    all_scenarios = json.load(f)

quality_scenarios = [s for s in all_scenarios if "reference" in s]
print(f"{len(quality_scenarios)} scenarios have a ground-truth reference:")
for s in quality_scenarios:
    expected = s.get("expected_tools") or ([s["expected_tool"]] if "expected_tool" in s else [])
    rag = " [RAG]" if s.get("is_rag") else ""
    print(f"  - {s['id']}: expects {expected}{rag}")

In [ ]:
# Run the quality evaluation: invoke each agent locally, score, push to Langfuse.
from utils.langfuse_metrics import get_latest_trace_metrics


def run_quality_eval(version_name, build_agent, scenarios):
    """Invoke the agent locally per scenario, score 3 RAGAS metrics, push to Langfuse."""
    rows = []
    print(f"\nQuality eval: {version_name}")

    for s in scenarios:
        # Fresh agent per scenario keeps conversations isolated.
        captured = ragas_eval.invoke_and_capture(build_agent(), s["query"])

        expected_tools = s.get("expected_tools") or ([s["expected_tool"]] if "expected_tool" in s else [])
        answer = ragas_eval.extract_answer(captured["response"])

        tca = ragas_eval.score_tool_call_accuracy(s["query"], captured["tool_calls"], expected_tools)
        ac = ragas_eval.score_answer_correctness(s["query"], answer, s["reference"], judge, embeddings)
        # Context precision only for RAG scenarios (get_technical_support).
        cp = None
        if s.get("is_rag"):
            cp = ragas_eval.score_context_precision(
                s["query"], s["reference"], captured["retrieved_contexts"], judge
            )

        # Push scores onto this version's latest Langfuse trace.
        trace = get_latest_trace_metrics(agent_name=version_name, wait_seconds=3, max_retries=3, timeout_seconds=60)
        trace_id = trace.get("trace_id")
        if trace_id:
            ragas_eval.push_scores_to_langfuse(
                trace_id,
                {"tool_call_accuracy": tca, "answer_correctness": ac, "context_precision": cp},
            )

        rows.append(
            {
                "scenario": s["id"],
                "tool_call_accuracy": tca,
                "answer_correctness": ac,
                "context_precision": cp,
            }
        )
        cp_str = f"{cp:.2f}" if cp is not None else "  - "
        print(f"  [{s['id']:<20}] TCA={tca:.2f}  AC={ac:.2f}  CP={cp_str}  (tools called: {captured['tool_calls']})")

    return rows


quality_results = {
    "v1-baseline": run_quality_eval("v1-baseline", build_v1_agent, quality_scenarios),
    "v4-optimized": run_quality_eval("v4-optimized", build_v4_agent, quality_scenarios),
}
print("\nQuality evaluation complete. Scores pushed to Langfuse.")

In [ ]:
# Summarize quality: average each metric per version (ignoring N/A where a metric doesn't apply).
def avg(rows, key):
    vals = [r[key] for r in rows if r.get(key) is not None]
    return sum(vals) / len(vals) if vals else None


summary_rows = []
for version_name, rows in quality_results.items():
    summary_rows.append(
        {
            "Version": version_name,
            "Tool Call Accuracy": avg(rows, "tool_call_accuracy"),
            "Answer Correctness": avg(rows, "answer_correctness"),
            "Context Precision (RAG)": avg(rows, "context_precision"),
        }
    )

df_quality = pd.DataFrame(summary_rows)
print("=" * 70)
print("QUALITY EVALUATION — baseline vs optimized")
print("=" * 70)
print(df_quality.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nSuccess criterion: optimized (v4) scores should hold at or above baseline (v1).")
print("Per-trace scores are visible in Langfuse under each trace's Scores tab,")
print("filterable by the 'ragas-eval' tag.")

## Cleanup (Optional)

Delete all deployed agents if you're done with the workshop.

In [ ]:
# # Uncomment to delete all customer-support agents and their ECR repositories
# from utils.runtime_helpers import cleanup_agents
# cleanup_agents(control_client, name_prefix="customer_support", region=region)

## Congratulations!

You've completed the Prompt Optimization Workshop!

**Key Takeaways:**
- Start with measurement (baseline metrics)
- Apply optimizations incrementally
- Validate quality at each step
- Use observability (Langfuse) to verify improvements

**Next Steps:**
- Apply these techniques to your own agents
- Explore additional Bedrock features
- Set up automated evaluation pipelines